# K-means: inspect one tiny iteration

Companion to HTB Academy Module 290, Section 10. Four **invented** one-dimensional points make assignments, centroid updates, WCSS, and one silhouette score inspectable by hand. This is a teaching implementation of the current section, not a production clustering library or an attack detector.

Read $a_i=\operatorname*{arg\,min}_k d(x_i,\mu_k)$ as *the cluster index k that minimizes the distance from point x sub i to centroid mu sub k*. Read $\mu_k=(1/|C_k|)\sum_{i\in C_k}x_i$ as *mu sub k is the mean of the points assigned to cluster C sub k*.

In [ ]:
points = (0.0, 1.0, 9.0, 10.0)
centroids = (0.0, 9.0)  # Two selected data points; K = 2.

def assign_nearest(values, centers):
    # In one dimension, |x-c| is Euclidean distance.
    # min over (distance, index) breaks exact ties toward the lower index.
    return tuple(min(range(len(centers)), key=lambda k: (abs(x - centers[k]), k))
                 for x in values)

def update_means(values, assignments, cluster_count):
    groups = [[x for x, assigned in zip(values, assignments) if assigned == k]
              for k in range(cluster_count)]
    assert all(groups), 'this simple study example does not handle empty clusters'
    return tuple(sum(group) / len(group) for group in groups)

first_assignment = assign_nearest(points, centroids)
updated_centroids = update_means(points, first_assignment, len(centroids))
second_assignment = assign_nearest(points, updated_centroids)
print('first assignment:', first_assignment)
print('updated centroids:', updated_centroids)
print('assignment after update:', second_assignment)
assert first_assignment == second_assignment == (0, 0, 1, 1)
assert updated_centroids == (0.5, 9.5)

## Check the squared-distance objective

Read $WCSS=\sum_k\sum_{i\in C_k}\lVert x_i-\mu_k\rVert_2^2$ as *sum, over each cluster and each of its points, the squared distance to that cluster's centroid*. The score is about compactness **within** clusters. For $K=1$, the centroid is the mean of all four points.

In [ ]:
def wcss(values, assignments, centers):
    # One-dimensional squared Euclidean distance: (x - center)^2.
    return sum((x - centers[k]) ** 2 for x, k in zip(values, assignments))

two_cluster_wcss = wcss(points, second_assignment, updated_centroids)
one_center = sum(points) / len(points)
one_cluster_wcss = wcss(points, (0, 0, 0, 0), (one_center,))
print(f'K=1: centroid={one_center:g}, WCSS={one_cluster_wcss:g}')
print(f'K=2: centroids={updated_centroids}, WCSS={two_cluster_wcss:g}')
assert one_cluster_wcss == 82.0
assert two_cluster_wcss == 1.0

## One silhouette value

For point `0`, read $a(0)$ as *average distance to the other points in its own cluster* and $b(0)$ as *smallest average distance to a different cluster*. Read $s(0)=[b(0)-a(0)]/\max(a(0),b(0))$ as *difference between other-cluster and own-cluster distances, divided by the larger distance*. This tiny example has two clusters of two points, avoiding singleton conventions.

In [ ]:
point = points[0]
own_neighbors = [x for index, x in enumerate(points)
                 if second_assignment[index] == second_assignment[0] and index != 0]
other_neighbors = [x for index, x in enumerate(points)
                   if second_assignment[index] != second_assignment[0]]
a = sum(abs(point - x) for x in own_neighbors) / len(own_neighbors)
b = sum(abs(point - x) for x in other_neighbors) / len(other_neighbors)
silhouette = (b - a) / max(a, b)
print(f'point 0: a={a:g}, b={b:g}, silhouette={silhouette:.3f}')
assert a == 1.0 and b == 9.5
assert abs(silhouette - 8.5 / 9.5) < 1e-12

K-means can produce clusters even if no meaningful groups exist. Its result depends on feature scaling, initialization, cluster shape, and outliers. A cluster number is **not** an attack label.